In [1]:
import sympy
from sympy import symbols, Eq, solve, pi

In [2]:
# --- Define Symbolic Parameters ---
# These are the inputs to our calculation. We are not using numbers,
# but symbols to build the formulas.

# Geometric Properties
a = symbols("a")  # Inner height of the hollow tube
b = symbols("b")  # Inner width of the hollow tube
t = symbols("t")  # Wall thickness of the tube
L = symbols("L")  # Length of one wing (from root to tip)

# Material Properties
E = symbols("E")  # Modulus of Elasticity (Young's Modulus) of the carbon fiber
S_ut = symbols("S_ut")  # Ultimate Tensile Strength of the material

# Loading Conditions
m = symbols("m")  # Total mass of the aircraft
G = symbols("G")  # G-force (load factor)
g = symbols("g")  # Acceleration due to gravity

# --- Geometric Calculations ---

# Outer dimensions are the inner dimensions plus twice the thickness
A = b + 2 * t  # Outer height
B = a + 2 * t  # Outer width

# Area Moment of Inertia (I) for a hollow rectangular tube.
# This property describes the beam's ability to resist bending.
# It's calculated by subtracting the moment of inertia of the inner
# "hollow" rectangle from the moment of inertia of the outer rectangle.
I_outer = (B * A**3) / 12
I_inner = (b * a**3) / 12
I = I_outer - I_inner

# --- Bending Moment Calculation ---

# The total lift must equal the aircraft's weight multiplied by the G-force.
# This lift is distributed over both wings.
F_total_lift = m * g * G

# The lift is assumed to be a uniformly distributed load (w) along the wings.
# The total length is 2*L (for two wings).
w = F_total_lift / (2 * L)

# The maximum bending moment (M_max) for a cantilever beam (one wing fixed at the fuselage)
# with a uniformly distributed load occurs at the root (the fixed end).
M_max = (w * L**2) / 2

# --- Stress Analysis ---

# Maximum Bending Stress (sigma_max)
# This is the stress experienced at the top and bottom surfaces of the spar at the wing root.
# 'c' is the distance from the neutral axis (center) to the outer fiber.
c = A / 2
sigma_max = (M_max * c) / I

# --- Buckling Analysis ---

# Critical Buckling Stress (sigma_cr) for a thin-walled rectangular tube in bending.
# This formula estimates the stress at which the spar's compression surface might buckle or wrinkle.
# K is a buckling coefficient, which for a long, simply supported plate is ~4.
K = 4
sigma_cr = (K * pi**2 * E) / (12 * (1 - 0.3**2)) * (t / b) ** 2  # Assuming Poisson's ratio of 0.3 for composite

# --- Factor of Safety Calculation ---

# Factor of Safety against Material Failure (Yielding/Breaking)
FS_yield = S_ut / sigma_max

# Factor of Safety against Buckling
FS_buckling = sigma_cr / sigma_max

# The overall Factor of Safety (FS) is the lower of the two.
# The structure is only as strong as its weakest failure mode.
FS = sympy.Min(FS_yield, FS_buckling)

In [3]:
# --- Display the Final Formulas ---
# The script can now print the symbolic formulas derived.

print("=" * 50)
print("Symbolic Wing Spar Safety Factor Equations")
print("=" * 50)

print("\n1. Maximum Bending Moment (M_max):")
sympy.pprint(Eq(symbols("M_max"), M_max), use_unicode=True)

print("\n2. Maximum Bending Stress (sigma_max):")
sympy.pprint(Eq(symbols("sigma_max"), sigma_max), use_unicode=True)

print("\n3. Critical Buckling Stress (sigma_cr):")
sympy.pprint(Eq(symbols("sigma_cr"), sigma_cr), use_unicode=True)

print("\n4. Factor of Safety against Material Yield (FS_yield):")
sympy.pprint(Eq(symbols("FS_yield"), FS_yield), use_unicode=True)

print("\n5. Factor of Safety against Buckling (FS_buckling):")
sympy.pprint(Eq(symbols("FS_buckling"), FS_buckling), use_unicode=True)

print("\n6. Overall Factor of Safety (FS):")
print("FS = Min(FS_yield, FS_buckling)")

Symbolic Wing Spar Safety Factor Equations

1. Maximum Bending Moment (M_max):
       G⋅L⋅g⋅m
Mₘₐₓ = ───────
          4   

2. Maximum Bending Stress (sigma_max):
                        ⎛b    ⎞         
                G⋅L⋅g⋅m⋅⎜─ + t⎟         
                        ⎝2    ⎠         
σₘₐₓ = ─────────────────────────────────
         ⎛   3                        3⎞
         ⎜  a ⋅b   (a + 2⋅t)⋅(b + 2⋅t) ⎟
       4⋅⎜- ──── + ────────────────────⎟
         ⎝   12             12         ⎠

3. Critical Buckling Stress (sigma_cr):
                          2    2
       0.366300366300366⋅π ⋅E⋅t 
σ_cr = ─────────────────────────
                   2            
                  b             

4. Factor of Safety against Material Yield (FS_yield):
                 ⎛   3                        3⎞
                 ⎜  a ⋅b   (a + 2⋅t)⋅(b + 2⋅t) ⎟
           4⋅Sᵤₜ⋅⎜- ──── + ────────────────────⎟
                 ⎝   12             12         ⎠
FS_yield = ─────────────────────────────────────
 

In [30]:
# --- Numerical Calculation Example ---
# You can change the values in the 'numerical_values' dictionary below
# to match your specific aircraft and materials.


def calculate_and_print_numerical(values, print_output=True):
    """
    Substitutes numerical values into the symbolic equations and prints the results.
    """
    if print_output:
        print("\n" + "=" * 50)
        print("Numerical Calculation Example")
        print("=" * 50)

        print("\nInput Parameters (all in SI units: m, kg, Pa, etc.):")

        # Pretty print the dictionary of inputs
        for key, value in values.items():
            print(f"- {str(key):<5}: {value}")

    try:
        # Substitute the dictionary of values into the symbolic expressions
        m_max_num = M_max.subs(values)
        sigma_max_num = sigma_max.subs(values)
        sigma_cr_num = sigma_cr.subs(values)
        fs_yield_num = FS_yield.subs(values)
        fs_buckling_num = FS_buckling.subs(values)

        # The Min function from sympy doesn't evaluate with .subs, so we calculate it manually
        fs_overall_num = min(fs_yield_num, fs_buckling_num)

        if print_output:
            print("\nCalculated Results:")
            print(f"- Maximum Bending Moment (M_max): {float(m_max_num):.2f} N*m")
            # Convert Pascals to MegaPascals (MPa) for readability
            print(f"- Maximum Bending Stress (sigma_max): {float(sigma_max_num) / 1e6:.2f} MPa")
            print(f"- Critical Buckling Stress (sigma_cr): {float(sigma_cr_num) / 1e6:.2f} MPa")
            print("-" * 35)
            print(f"- Factor of Safety (Yielding): {float(fs_yield_num):.2f}")
            print(f"- Factor of Safety (Buckling): {float(fs_buckling_num):.2f}")
            print("-" * 35)
            print(f"-> Overall Factor of Safety: {float(fs_overall_num):.2f}")
            print("=" * 50)
        else:
            return fs_overall_num, fs_yield_num, fs_buckling_num

    except Exception as e:
        print(f"\nAn error occurred during calculation: {e}")
        print("Please ensure all input values are valid numbers.")


In [39]:
# --- DEFINE YOUR NUMBERS HERE ---
# Example values for a hypothetical light aircraft / large drone.
# All units must be in standard SI (meters, kilograms, seconds, Pascals).

inch_to_m = 0.0254  # 1 inch = 0.0254 meters
lbs_to_kg = 0.453592  # 1 pound = 0.453592 kilograms
msi_to_pa = 6.89476 * 1e9

numerical_values = {
    a: 0.75 * inch_to_m,  # inner height
    b: 0.75 * 2 * inch_to_m,  # inner width
    t: 0.0625 * inch_to_m,  # wall thickness
    L: 2.5 * 12 * inch_to_m,  # 1/2 wing length
    E: 33 * msi_to_pa,  # Modulus of Elasticity for carbon fiber
    S_ut: 500 * 1e6,  # Ultimate Tensile Strength
    m: 20 * lbs_to_kg,  # total aircraft mass
    G: 6,  # 6G load factor
    g: 9.81,  # 9.81 m/s^2 acceleration of gravity
}

# Run the numerical calculation with the values defined above
calculate_and_print_numerical(numerical_values)


Numerical Calculation Example

Input Parameters (all in SI units: m, kg, Pa, etc.):
- a    : 0.019049999999999997
- b    : 0.038099999999999995
- t    : 0.0015875
- L    : 0.762
- E    : 227527080000.0
- S_ut : 500000000.0
- m    : 9.07184
- G    : 6
- g    : 9.81

Calculated Results:
- Maximum Bending Moment (M_max): 101.72 N*m
- Maximum Bending Stress (sigma_max): 19.39 MPa
- Critical Buckling Stress (sigma_cr): 1428.06 MPa
-----------------------------------
- Factor of Safety (Yielding): 25.79
- Factor of Safety (Buckling): 73.66
-----------------------------------
-> Overall Factor of Safety: 25.79
